# Projeto — Análise de Preços de Imóveis

**Dataset:** Housing Prices Dataset (King County)

Notebook organizado segundo as fases e questões do desafio.

## 🔹 FASE 1 — Limpeza e Padronização

### 1. Consolidação e Tipagem
Conversão das colunas para os tipos adequados: data para datetime, áreas de pés quadrados para metros quadrados e código postal para texto.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

#import plotly.io as pio
#pio.renderers.default = "vscode"

In [ ]:
df = pd.read_csv('Housing.csv')

In [ ]:
df['date'] = pd.to_datetime(df['date'])

In [ ]:
#Convertendo a Para Metros Quadrados

converter = ['sqft_living' , 'sqft_lot', 'sqft_above', 'sqft_basement', 'sqft_living15', 'sqft_lot15']

for i in converter:
    df[i]= df[i] * 0.092903

In [ ]:
df['zipcode'] = df['zipcode'].astype(str)

In [ ]:
df.info()

### 2. Tratamento de Valores Ausentes
Verificação de valores nulos por coluna. O dataset não apresenta ausências, portanto nenhum registro precisou ser removido.

In [ ]:
df.isna().sum()

### 3. Padronização de Variáveis Categóricas
Renomeação das colunas para português, padronizando a nomenclatura das variáveis de localização e tipo do imóvel.

In [ ]:

traducao_colunas = {
    'id': 'id_imovel',
    'date': 'data',
    'price': 'preco',
    'bedrooms': 'quartos',
    'bathrooms': 'banheiros',
    'sqft_living': 'm2_area_habitavel',    
    'sqft_lot': 'm2_terreno',
    'floors': 'andares',
    'waterfront': 'vista_mar_rio',         
    'view': 'nota_vista',                  
    'condition': 'condicao_imovel',        
    'grade': 'nota_design',                
    'sqft_above': 'm2_acima_solo',          
    'sqft_basement': 'm2_porao',          
    'yr_built': 'ano_construcao',
    'yr_renovated': 'ano_renovacao',
    'zipcode': 'codigo_postal',
    'lat': 'latitude',
    'long': 'longitude',
    'sqft_living15': 'm2_area_vizinhos_15', 
    'sqft_lot15': 'm2_terreno_vizinhos_15' 
}

df_traduzido = df.rename(columns=traducao_colunas)

print(df_traduzido.columns)

## 🔹 FASE 2 — Consultas e Agregações

### 4. Análise Histórica de Preços
Preço médio por ano e por mês. O período do dataset (mai/2014 a mai/2015) cobre apenas 13 meses, então a comparação anual reflete meses parciais.

In [ ]:
df_traduzido['ano'] = df_traduzido['data'].dt.year
df_traduzido['mes'] = df_traduzido['data'].dt.month

In [ ]:
df_media_anual = df_traduzido.groupby('ano')['preco'].mean().reset_index().round(2)

In [ ]:
df_media_anual

In [ ]:
media_mensal = df_traduzido.groupby(['ano','mes'])['preco'].mean().reset_index().round(2)
media_mensal

### 5. Análise Localizada
Preço médio, mínimo e máximo de um bairro (CEP) específico dentro de um intervalo de anos definido.

In [ ]:
df_traduzido['codigo_postal'].value_counts().reset_index().head()

In [ ]:
bairro_escolhido ='98103'
#intevalo de anos:
ano_ini , ano_fim = 2014, 2015


df_bairro = df_traduzido[(df_traduzido['codigo_postal'] == bairro_escolhido) & (df_traduzido['ano'].between(ano_ini, ano_fim)) ]

print(f'O preço médio do cep {bairro_escolhido} é R${df_bairro['preco'].mean():.2f}, o maior preco é {df_bairro['preco'].max():.2f} o menor é {df_bairro['preco'].min()} ')

### 6. Ranking de Bairros
Cinco bairros com maior preço médio. Os valores mais altos concentram-se em regiões nobres de King County (ex.: 98039 - Medina).

In [ ]:
media_preco_bairro = (df_traduzido.groupby('codigo_postal')['preco'].mean()
.sort_values(ascending=False)
.reset_index()
.rename(columns= ({'preco' : 'Media_preco','codigo_postal' :'Bairro' }))
.round(2)
)
media_preco_bairro


In [ ]:
top5 = media_preco_bairro.head(5)
top5

In [ ]:
fig_top5 = px.bar(
    top5,
    x = 'Media_preco',
    y = 'Bairro',
    orientation='h',
    text_auto = '.3s',
    color = 'Media_preco',
    title ='Top 5 bairros com maiores preços médios'
)
fig_top5.update_yaxes(type = 'category')
fig_top5.show()


## 🔹 FASE 3 — Estatística e Outliers

### 7. Identificação de Outliers
Boxplot e histograma da distribuição de preços. Os valores atípicos correspondem a imóveis de luxo reais (mansões, frente-mar), não a inconsistências, e por isso foram mantidos.

In [ ]:
fig_outlier= px.box(df_traduzido, y='preco' , title= 'Distribuição de Preços com Outliers', points ='outliers')
fig_outlier.show()


In [ ]:
fig_hist = px.histogram(df_traduzido , x = 'preco' , nbins= 100 , title ='Distribuição de Preços')
fig_hist.show()

## 🔹 FASE 4 — Visualização e Análise Exploratória

### 8. Relação entre Área e Preço
Dispersão entre área habitável e preço, evidenciando a tendência de crescimento (não perfeitamente linear).

In [ ]:

fig_area = px.scatter(
    df_traduzido, x="m2_area_habitavel", y="preco", title="Relação preço por área"
)
fig_area.show()

### 9. Evolução Temporal dos Preços
Preço médio mês a mês ao longo de todo o período disponível.

In [ ]:

media_mensal['data_formatada'] = media_mensal['ano'].astype(str) + '-' + media_mensal['mes'].astype(str)

fig_preco = px.line(
    media_mensal,
    x='data_formatada',
    y='preco',
    title='Evolução do Preço ao Longo do Tempo',
    labels={'data_formatada': 'Período (Ano-Mês)', 'preco': 'Preço ($)'}
)
fig_preco.update_xaxes(type='category')

fig_preco.show()

## 🔹 FASE 5 — Engenharia de Atributos e Correlações

### 11. Criação de Novas Variáveis
Atributos derivados: densidade de ocupação do lote, idade efetiva (considerando reformas), total de cômodos e indicador de renovação.

In [ ]:
df_traduzido['habitavel_por_terreno'] = df_traduzido['m2_area_habitavel'] / df_traduzido['m2_terreno']
df_traduzido['idade_efetiva'] = df_traduzido['ano'] - df_traduzido[['ano_construcao','ano_renovacao']].max(axis=1)
df_traduzido['total_comodos'] = df_traduzido['quartos'] + df_traduzido['banheiros']
df_traduzido['foi_renovado'] = (df_traduzido['ano_renovacao'] > 0).astype(int)




### 12. Análise de Correlação
Correlação de Pearson (relações lineares e redundância entre features) comparada com Spearman (relações monotônicas), para identificar as variáveis mais influentes e evitar descartar relações não lineares.

In [ ]:
cols_tirar = ['id_imovel', 'data' , 'codigo_postal']
df_correlacao_pearson = df_traduzido.drop(columns= cols_tirar).corr('pearson')


df_correlacao_pearson

In [ ]:
corr_preco_pearson = df_correlacao_pearson['preco'].drop('preco').abs()
corr_preco_spearman = df_traduzido.corr('spearman')['preco'].sort_values(ascending=False).drop('preco').abs()


In [ ]:
comparacao = pd.DataFrame({
    'pearson': corr_preco_pearson,
    'spearman': corr_preco_spearman
})
comparacao['diferenca'] = (comparacao['spearman'] - comparacao['pearson']).round(3)
comparacao['pearson'] = comparacao['pearson'].round(3)
comparacao['spearman'] = comparacao['spearman'].round(3)

comparacao = comparacao.reindex(
    comparacao['spearman'].abs().sort_values(ascending=False).index
)
print(comparacao)

In [ ]:
fig_heatmap = px.imshow(
    df_correlacao_pearson,
    text_auto='.2f',            # escreve o valor em cada célula
    aspect='auto',
    color_continuous_scale='RdBu_r',   # vermelho = +, azul = -
    zmin=-1, zmax=1,            # fixa a escala de cor de -1 a 1
    title='Mapa de calor — correlação de Pearson entre variáveis'
                        )
fig_heatmap.update_layout(height=800, width=900)
fig_heatmap.show()

### 13. Transformação de Variáveis Categóricas
A única categórica relevante para a modelagem seria o código postal (70 níveis, alta cardinalidade). Optou-se por descartá-la, pois a informação de localização é preservada de forma contínua pelas variáveis **latitude** e **longitude** — evitando a explosão dimensional que um one-hot encoding de 70 colunas causaria. As demais variáveis já são numéricas, dispensando codificação.

## 🔹 FASE 6 — Machine Learning

### 14. Modelo de Previsão de Preço
Regressão polinomial regularizada (L2) implementada manualmente, com padronização z-score e descida de gradiente. A modelagem segue por experimentos incrementais, medindo o efeito de cada decisão.

#### Preparação: seleção de features, split e normalização (z-score)
Remoção de duplicatas de revenda (evita vazamento), seleção das 9 features principais (sem geografia, nesta primeira configuração) e padronização com estatísticas calculadas apenas no treino.

In [ ]:
from sklearn.model_selection import train_test_split

features = [ 'm2_area_habitavel', 'nota_design', 'm2_area_vizinhos_15',
    'total_comodos', 'nota_vista', 'm2_porao',
    'habitavel_por_terreno', 'idade_efetiva', 'vista_mar_rio']

df_modelo = df_traduzido.sort_values('data').drop_duplicates('id_imovel', keep = 'last')
X = df_modelo[features].values
y = df_modelo['preco'].values


X_train, X_test,y_train,y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42
)
mu = X_train.mean(axis = 0)
sigma =X_train.std(axis= 0)

X_train_norm = (X_train - mu) / sigma
X_test_norm = (X_test - mu) / sigma

print("Média treino :", X_train_norm.mean(axis=0).round(3))
print("Desvio treino :", X_train_norm.std(axis=0).round(3))
print("\nMédia teste :", X_test_norm.mean(axis=0).round(3))

#### Termos polinomiais (potências) e função de custo
Geração dos termos polinomiais e definição da descida de gradiente com regularização L2. O alvo também é padronizado para estabilizar os gradientes.

In [ ]:
def gerar_termos_polinomiais(X,grau):
    lista_termos = [X]
    for g in range(2 , grau+1):
        lista_termos.append(X ** g)
    return np.concatenate(lista_termos, axis =1)

X_train_poly = gerar_termos_polinomiais(X_train_norm , grau = 2)
X_test_poly  = gerar_termos_polinomiais(X_test_norm, grau =2)

print(f"Antes:  {X_train_norm.shape[1]} features")
print(f"Depois (grau 2): {X_train_poly.shape[1]} features")

In [ ]:
def descida_gradiente(X, y, alpha=0.01, lambda_reg=0.1, n_iter=1000):
    m, n = X.shape
    w = np.zeros(n)
    b = 0.0
    historico_custo = []
    for i in range(n_iter):

        y_pred = X @ w + b
        erro = y_pred - y

        grad_w = (X.T @ erro) / m + (lambda_reg / m) * w

        grad_b = erro.mean()

        w = w - alpha * grad_w
        b = b - alpha * grad_b

        custo = (erro**2).mean() / 2 + (lambda_reg / (2 * m)) * (w**2).sum()

        historico_custo.append(custo)
        
    return w, b, historico_custo

In [ ]:
mu_y = y_train.mean()
sigma_y = y_train.std()

y_train_norm = (y_train - mu_y) / sigma_y
y_test_norm  = (y_test - mu_y) / sigma_y


#### Experimento 1 — Base (sem geografia, só potências)
Treino, curva de convergência do custo e avaliação em dólares (MAE, RMSE, R²).

In [ ]:
# Executa o treinamento
w, b, historico = descida_gradiente(
    X_train_poly, y_train_norm,
    alpha=0.01, lambda_reg=0.1, n_iter=1000
)

# Plota a curva de convergência
fig_custo = px.line(y=historico, title='Convergência do custo')
fig_custo.update_xaxes(title='Iteração')
fig_custo.update_yaxes(title='Custo (escala normalizada)')
fig_custo.show()

print(f"Custo inicial: {historico[0]:.4f}")
print(f"Custo final:   {historico[-1]:.4f}")

In [ ]:
# Previsões no teste (em escala normalizada)
y_pred_norm = X_test_poly @ w + b

# Desfaz a padronização: volta para dólares
y_pred = y_pred_norm * sigma_y + mu_y

# Métricas comparando com o preço real (y_test em dólares)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE:  US${mae:,.2f}")
print(f"RMSE: US${rmse:,.2f}")
print(f"R²:   {r2:.4f}")

#### Experimento 2 — Adicionando latitude e longitude
Mesmo pipeline, agora com as coordenadas geográficas, para medir o impacto da localização no desempenho.

In [ ]:
# EXPERIMENTO 2 — com latitude e longitude
features_geo = [
    'm2_area_habitavel', 'nota_design', 'm2_area_vizinhos_15',
    'total_comodos', 'nota_vista', 'm2_porao',
    'habitavel_por_terreno', 'idade_efetiva', 'vista_mar_rio',
    'latitude', 'longitude'          # <- as duas novas
]

X_geo = df_modelo[features_geo]
y_geo = df_modelo['preco']

# Split (mesmo random_state → mesma divisão, comparação justa)
X_train_g, X_test_g, y_train_g, y_test_g = train_test_split(
    X_geo, y_geo, test_size=0.2, random_state=42
)

# Normalização z-score (estatísticas só do treino)
mu_g, sigma_g = X_train_g.mean(), X_train_g.std()
X_train_g_norm = (X_train_g - mu_g) / sigma_g
X_test_g_norm  = (X_test_g - mu_g) / sigma_g

# Padroniza o alvo
mu_yg, sigma_yg = y_train_g.mean(), y_train_g.std()
y_train_g_norm = (y_train_g - mu_yg) / sigma_yg

# Termos polinomiais grau 2
X_train_g_poly = gerar_termos_polinomiais(X_train_g_norm.values, grau=2)
X_test_g_poly  = gerar_termos_polinomiais(X_test_g_norm.values, grau=2)

# Treina
w_g, b_g, hist_g = descida_gradiente(
    X_train_g_poly, y_train_g_norm.values,
    alpha=0.01, lambda_reg=0.1, n_iter=1000
)

# Prevê e reverte para dólares
y_pred_g = (X_test_g_poly @ w_g + b_g) * sigma_yg + mu_yg

# Métricas
mae_g = mean_absolute_error(y_test_g, y_pred_g)
rmse_g = np.sqrt(mean_squared_error(y_test_g, y_pred_g))
r2_g = r2_score(y_test_g, y_pred_g)

print("COM geografia:")
print(f"MAE:  US${mae_g:,.2f}")
print(f"RMSE: US${rmse_g:,.2f}")
print(f"R²:   {r2_g:.4f}")
print(f"\nSem geografia era: R² 0.6896 | MAE US$135,743")

#### Experimento 3 — Termos de interação (grau 2 completo)
Inclusão dos produtos cruzados entre features (efeitos combinados), seguida do diagnóstico de sobreajuste (treino vs. teste).

In [ ]:
from itertools import combinations_with_replacement

def gerar_termos_interacao(X, grau=2):
    """
    Gera termos polinomiais COM interações, até o grau dado.
    Grau 2: features originais + quadrados + produtos cruzados (xi * xj).
    """
    n_amostras, n_features = X.shape
    termos = [X]   # grau 1: as features originais

    # Para cada grau de 2 até 'grau', gera todas as combinações de features
    for g in range(2, grau + 1):
        for combo in combinations_with_replacement(range(n_features), g):
            # combo é uma tupla de índices, ex: (0,0)=x0², (0,1)=x0*x1
            termo = np.ones(n_amostras)
            for idx in combo:
                termo = termo * X[:, idx]
            termos.append(termo.reshape(-1, 1))

    return np.concatenate(termos, axis=1)

In [ ]:
# EXPERIMENTO 3 — com geografia E interações
X_train_int = gerar_termos_interacao(X_train_g_norm.values, grau=2)
X_test_int  = gerar_termos_interacao(X_test_g_norm.values, grau=2)

print(f"Sem interações (grau 2): {X_train_g_poly.shape[1]} termos")
print(f"Com interações (grau 2): {X_train_int.shape[1]} termos")

# Treina
w_int, b_int, hist_int = descida_gradiente(
    X_train_int, y_train_g_norm.values,
    alpha=0.01, lambda_reg=0.1, n_iter=1000
)

# Prevê e reverte para dólares
y_pred_int = (X_test_int @ w_int + b_int) * sigma_yg + mu_yg

# Métricas
mae_int = mean_absolute_error(y_test_g, y_pred_int)
rmse_int = np.sqrt(mean_squared_error(y_test_g, y_pred_int))
r2_int = r2_score(y_test_g, y_pred_int)

print(f"\nCOM interações:")
print(f"MAE:  US${mae_int:,.2f}")
print(f"RMSE: US${rmse_int:,.2f}")
print(f"R²:   {r2_int:.4f}")
print(f"\nComparação:")
print(f"  Só potências + geo: R² 0.7396")
print(f"  Com interações:     R² {r2_int:.4f}")

In [ ]:
# R² no treino vs no teste — diagnóstico de overfitting
y_pred_train_int = (X_train_int @ w_int + b_int) * sigma_yg + mu_yg

r2_train = r2_score(y_train_g, y_pred_train_int)
r2_test = r2_score(y_test_g, y_pred_int)

print(f"R² treino: {r2_train:.4f}")
print(f"R² teste:  {r2_test:.4f}")
print(f"Diferença: {r2_train - r2_test:.4f}")

#### Experimento 4 — Grau 3 e regularização
Aumento da complexidade para grau 3 (363 termos). Foi necessário renormalizar os termos polinomiais para a descida convergir. Avaliação do ganho e teste do efeito do parâmetro de regularização λ.

In [ ]:
# Gera termos grau 3
X_train_g3 = gerar_termos_interacao(X_train_g_norm.values, grau=3)
X_test_g3  = gerar_termos_interacao(X_test_g_norm.values, grau=3)

# RENORMALIZA os termos polinomiais (estatísticas só do treino)
mu_poly = X_train_g3.mean(axis=0)
sigma_poly = X_train_g3.std(axis=0)
sigma_poly[sigma_poly == 0] = 1        # evita divisão por zero em termos constantes

X_train_g3 = (X_train_g3 - mu_poly) / sigma_poly
X_test_g3  = (X_test_g3 - mu_poly) / sigma_poly

print(f"Grau 3: {X_train_g3.shape[1]} termos, renormalizados")

# Agora a descida converge com alpha normal
w_g3, b_g3, hist_g3 = descida_gradiente(
    X_train_g3, y_train_g_norm.values,
    alpha=0.01, lambda_reg=0.1, n_iter=3000
)

# Curva de custo para confirmar convergência
fig = px.line(y=hist_g3, title='Convergência grau 3')
fig.show()
print("Custo final:", hist_g3[-1])

In [ ]:
y_pred_g3       = (X_test_g3  @ w_g3 + b_g3) * sigma_yg + mu_yg
y_pred_g3_train = (X_train_g3 @ w_g3 + b_g3) * sigma_yg + mu_yg

r2_test_g3  = r2_score(y_test_g, y_pred_g3)
r2_train_g3 = r2_score(y_train_g, y_pred_g3_train)
mae_g3  = mean_absolute_error(y_test_g, y_pred_g3)
rmse_g3 = np.sqrt(mean_squared_error(y_test_g, y_pred_g3))

print(f"Grau 3:")
print(f"MAE:  US${mae_g3:,.2f}")
print(f"RMSE: US${rmse_g3:,.2f}")
print(f"R² teste:  {r2_test_g3:.4f}")
print(f"R² treino: {r2_train_g3:.4f}")
print(f"Diferença treino-teste: {r2_train_g3 - r2_test_g3:.4f}")
print(f"\nGrau 2 tinha: R² teste 0.7940 | diferença -0.0019")


In [ ]:
# Grau 3 com mais regularização — fecha a lacuna treino-teste?
for lam in [0.1, 1.0, 10.0]:
    w_t, b_t, _ = descida_gradiente(X_train_g3, y_train_g_norm.values,
                                     alpha=0.01, lambda_reg=lam, n_iter=3000)
    yp_test  = (X_test_g3  @ w_t + b_t) * sigma_yg + mu_yg
    yp_train = (X_train_g3 @ w_t + b_t) * sigma_yg + mu_yg
    r2te = r2_score(y_test_g, yp_test)
    r2tr = r2_score(y_train_g, yp_train)
    print(f"λ={lam:5.1f} | R² teste {r2te:.4f} | R² treino {r2tr:.4f} | dif {r2tr-r2te:.4f}")

### 16. Agrupamento de Imóveis ou Bairros
*(A implementar — clustering de bairros por comportamento de preços.)*

## 🔹 FASE 7 — Deep Learning

### 17. Preparação dos Dados para Rede Neural
*(A implementar.)*

### 18. Construção da Rede Neural
*(A implementar.)*

### 19. Treinamento e Avaliação do Modelo
*(A implementar — treino, avaliação e análise de sobre/subajuste.)*